In [19]:
import os 
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import ( 
        Dataset, 
        DataLoader, 
        random_split, 
        WeightedRandomSampler
    )
import torchvision.models as models 
import torchvision.transforms.functional as TF 
from torchvision.transforms import ColorJitter 
from PIL import Image 

SEED = 42 
random.seed(SEED) 
np.random.seed(SEED) 
torch.manual_seed(SEED) 

if torch.cuda.is_available(): 
    torch.cuda.manual_seed_all(SEED) 

torch.backends.cudnn.deterministic = True 
torch.backends.cudnn.bechnmark = False 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 

print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device : cuda
PyTorch: 2.11.0+cu128
Tesla T4


In [20]:
import os 
import random 
import torch
from PIL import Image 
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler 
import torchvision.transforms.functional as TF 
from torchvision.transforms import ColorJitter

SEED = 42 
random.seed(SEED) 
torch.manual_seed(SEED)
if torch.cuda.is_available(): 
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

class CAT2000Dataset(Dataset): 
    OFFICIAL_CATEGORIES = [
        "Action", "Affective", "Art", "Cartoon", "DigitalArt", 
        "Everyday", "Fractal", "Indoor", "Inverted", "LightNormal", 
        "LineDrawing", "Loud", "LowResolution", "Natural", "Object",
        "Outdoor", "Photo", "Random", "Social", "Synthetic" 
    ]
    
    def __init__(self, root_dir, augment=False, test_mode=False): 
        self.root_dir = os.path.abspath(root_dir)
        self.stimuli_dir = os.path.join(root_dir, "Stimuli")
        self.maps_dir = os.path.join(root_dir, "FixationMaps")
        self.test_mode = test_mode

        if test_mode:
            self.maps_dir = None
        else:
            self.maps_dir = os.path.join(root_dir, "FixationMaps")
        self.augment = augment 
        self.color_jitter = ColorJitter( 
            brightness = 0.15, 
            contrast = 0.15,  
            saturation = 0.10, 
            hue = 0.02
                                        )
        self.cat_to_idx = { 
            cat: idx 
            for idx, cat in enumerate(self.OFFICIAL_CATEGORIES)
            }
        self.samples = [] 
        self._index_dataset() 
        
    def _index_dataset(self): 
        for cat in self.OFFICIAL_CATEGORIES: 
            
            stim_folder = os.path.join(self.stimuli_dir, cat)
            
            if not os.path.exists(stim_folder): 
                continue
            
            if self.test_mode: 
                map_folder = os.path.join(stim_folder, "Output")
            else: 
                map_folder = os.path.join(self.maps_dir, cat)
            
            for filename in sorted(os.listdir(stim_folder)): 
                if not filename.lower().endswith((".jpg", ".jpeg", ".png")): 
                    continue
                
                image_path = os.path.join(stim_folder, filename) 
                
                if self.test_mode:
                    stem = os.path.splitext(filename)[0]
                    map_path = os.path.join(
                        map_folder,
                        f"{stem}_SaliencyMap.jpg"
                    )
                else:
                    map_path = os.path.join(map_folder, filename)

                if os.path.exists(map_path):
                    self.samples.append({
                        "image": image_path,
                        "map": map_path,
                        "cat_idx": self.cat_to_idx[cat]
                    })
        
    def __len__(self): 
        return len(self.samples)
    
    def __getitem__(self, idx): 
        sample = self.samples[idx] 
        image = Image.open(sample["image"]).convert("RGB")
        fix_map = Image.open(sample["map"]).convert("L")
        
        if self.augment: 
            if random.random() < 0.5: 
                image = TF.hflip(image)
                fix_map = TF.hflip(fix_map)
                
            image = self.color_jitter(image)
        
        image = TF.resize(
            image,
            (224, 224),
            interpolation=TF.InterpolationMode.BILINEAR
        )
        fix_map = TF.resize( 
            fix_map, 
            (224, 224), 
            interpolation=TF.InterpolationMode.BILINEAR
            )
        
        image_tensor = TF.to_tensor(image)
        fix_tensor = TF.to_tensor(fix_map)
        
        fix_tensor = (
        fix_tensor - fix_tensor.min()
        ) / (
        fix_tensor.max() - fix_tensor.min() + 1e-6
        )
    
        image_tensor = TF.normalize( 
                image_tensor, 
                mean=[0.485, 0.456, 0.406], 
                std = [0.229, 0.224, 0.225])
    
        return image_tensor, fix_tensor, sample["cat_idx"]
    
def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)
    
base_path = "/content/drive/MyDrive/CAT2000/trainSet"
full_dataset = CAT2000Dataset( 
            root_dir=base_path, 
            augment=True
            )
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size 

generator = torch.Generator().manual_seed(SEED)

train_dataset, val_dataset = random_split( 
                full_dataset,  
                [train_size, val_size], 
                generator= generator
                )
train_labels = [
    full_dataset.samples[i]["cat_idx"]
    for i in train_dataset.indices
]
class_counts = torch.bincount(torch.tensor(train_labels, dtype=torch.long))
class_weights = 1.0 / class_counts.float() 

sample_weights = [ 
        class_weights[label] 
        for label in train_labels
]
sampler = WeightedRandomSampler( 
            sample_weights, 
            num_samples = len(sample_weights),
            replacement=True
        )

production_loader = DataLoader( 
        train_dataset, 
        batch_size=8, 
        sampler= sampler,
        worker_init_fn=seed_worker
    )
validation_loader = DataLoader( 
        val_dataset, 
        batch_size=8, 
        shuffle=False,
        worker_init_fn=seed_worker
)

print(f"Total images      : {len(full_dataset)}")
print(f"Training images   : {len(train_dataset)}")
print(f"Validation images : {len(val_dataset)}")
print(f"Categories        : {len(full_dataset.OFFICIAL_CATEGORIES)}")
print(f"Random seed      : {SEED}")


Total images      : 1200
Training images   : 960
Validation images : 240
Categories        : 20
Random seed      : 42


In [21]:
import torch.nn as nn 
import torchvision.models as models 

backbone = models.resnet50(weights=None)
modules = list(backbone.children())[:-2]
feature_extractor = nn.Sequential(*modules).to(device)
feature_extractor.train() 
print("--> Scratch ResNet-50 initialized successfully.")
print("--> Backbone parameters are trainable.")

--> Scratch ResNet-50 initialized successfully.
--> Backbone parameters are trainable.


In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SaliencyDecoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(2048, 512, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(512, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):

        x = F.relu(self.conv1(x))

        x = F.interpolate(
            x,
            scale_factor=4,
            mode="bilinear",
            align_corners=False
        )

        x = F.relu(self.conv2(x))

        x = F.interpolate(
            x,
            scale_factor=4,
            mode="bilinear",
            align_corners=False
        )

        x = self.conv3(x)

        x = F.interpolate(
            x,
            scale_factor=2,
            mode="bilinear",
            align_corners=False
        )

        return torch.sigmoid(x)

decoder = SaliencyDecoder().to(device)

print("--> Saliency decoder initialized.")

--> Saliency decoder initialized.
